## Pyspark vs Pandas

| 구분            | **Pandas**                   | **PySpark (Spark DataFrame)**     |
| ------------- | ---------------------------- | --------------------------------- |
| **기본 목적**     | 소규모 데이터 분석                   | 대용량(수 GB~TB) 분산 처리                |
| **데이터 처리 위치** | 메모리(RAM) 안에서                 | 여러 서버(클러스터)에 분산                   |
| **속도**        | 단일 CPU 기반 (작은 데이터 빠름)        | 병렬 처리 (큰 데이터 효율적)                 |
| **데이터 크기 한계** | 메모리에 맞는 크기까지만                | 거의 무제한 (디스크·클러스터 기반)              |
| **언어 스타일**    | Pythonic (직관적)               | SQL 스타일 + 함수 체인형                  |
| **적합한 용도**    | EDA, 통계분석, 머신러닝 전처리          | 빅데이터 분석, 로그 처리, ETL 파이프라인         |
| **대표 함수**     | `df.groupby()`, `df.apply()` | `df.groupBy()`, `df.selectExpr()` |
| **예시 라이브러리**  | NumPy, scikit-learn          | Hadoop, Hive, Spark MLlib         |


- pandas
     - 수천~수만 행 데이터는 RAM 안에서 바로 계산 가능
     - 하지만 10GB 이상이면 “MemoryError” 발생 가능 ⚠️

- pyspark
     - 데이터가 100GB든 1TB든, Spark가 여러 서버(노드) 로 나눠서 병렬 처리
     - 로컬에서도 작은 클러스터처럼 흉내 가능

pandas에서 pyspark로 확장 가능(반대도 가능)

```python
pandas_df = df.toPandas()   # Spark → Pandas
spark_df = spark.createDataFrame(pandas_df)   # Pandas → Spark
```

- pyspark는 java기반 engine이기 때문에 openjdk 설치 필요

In [7]:
!apt-get install -y openjdk-11-jdk-headless -qq
!pip install -q pyspark


Selecting previously unselected package java-common.
(Reading database ... 125082 files and directories currently installed.)
Preparing to unpack .../java-common_0.72build2_all.deb ...
Unpacking java-common (0.72build2) ...
Selecting previously unselected package libpcsclite1:amd64.
Preparing to unpack .../libpcsclite1_1.9.5-3ubuntu1_amd64.deb ...
Unpacking libpcsclite1:amd64 (1.9.5-3ubuntu1) ...
Selecting previously unselected package openjdk-11-jre-headless:amd64.
Preparing to unpack .../openjdk-11-jre-headless_11.0.28+6-1ubuntu1~22.04.1_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.28+6-1ubuntu1~22.04.1) ...
Selecting previously unselected package ca-certificates-java.
Preparing to unpack .../ca-certificates-java_20190909ubuntu1.2_all.deb ...
Unpacking ca-certificates-java (20190909ubuntu1.2) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.28+6-1ubuntu1~22.04.1_amd64.deb ...
Unpacking openj

In [3]:
from pyspark.sql import SparkSession

환경변수 설정

In [5]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

spark session 지정

In [8]:
spark = SparkSession.builder \
    .appName("Colab EHR Demo") \
    .getOrCreate()

spark

In [9]:
import os, glob

glob.glob("/content/sample_data/*")

['/content/sample_data/anscombe.json',
 '/content/sample_data/README.md',
 '/content/sample_data/mnist_test.csv',
 '/content/sample_data/california_housing_test.csv',
 '/content/sample_data/california_housing_train.csv',
 '/content/sample_data/mnist_train_small.csv']

In [19]:
path = "/content/sample_data/california_housing_train.csv"

In [20]:
df = spark.read.csv(path, header=True, inferSchema=True)

In [21]:
df.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|  -114.31|   34.19|              15.0|     5612.0|        1283.0|    1015.0|     472.0|       1.4936|           66900.0|
|  -114.47|    34.4|              19.0|     7650.0|        1901.0|    1129.0|     463.0|         1.82|           80100.0|
|  -114.56|   33.69|              17.0|      720.0|         174.0|     333.0|     117.0|       1.6509|           85700.0|
|  -114.57|   33.64|              14.0|     1501.0|         337.0|     515.0|     226.0|       3.1917|           73400.0|
|  -114.57|   33.57|              20.0|     1454.0|         326.0|     624.0|     262.0|        1.925|           65500.0|
+---------+--------+----

In [22]:
df.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)



In [11]:
df.columns

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income',
 'median_house_value']

In [13]:
print("Rows:", df.count())
print("Cols:", len(df.columns))

Rows: 17000
Cols: 9


In [14]:
df.describe().show()

+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+
|summary|          longitude|          latitude|housing_median_age|      total_rooms|   total_bedrooms|        population|       households|     median_income|median_house_value|
+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+
|  count|              17000|             17000|             17000|            17000|            17000|             17000|            17000|             17000|             17000|
|   mean|-119.56210823529375|  35.6252247058827| 28.58935294117647|2643.664411764706|539.4108235294118|1429.5739411764705|501.2219411764706| 3.883578100000021|207300.91235294117|
| stddev| 2.0051664084260357|2.1373397946570867|12.586936981660406|2179.947071452777|421.4994515798648| 1

In [16]:
df.select("median_income", "median_house_value").show(5)

+-------------+------------------+
|median_income|median_house_value|
+-------------+------------------+
|       1.4936|           66900.0|
|         1.82|           80100.0|
|       1.6509|           85700.0|
|       3.1917|           73400.0|
|        1.925|           65500.0|
+-------------+------------------+
only showing top 5 rows



In [17]:
df.selectExpr("avg(median_house_value) as avg_house_value").show()

+------------------+
|   avg_house_value|
+------------------+
|207300.91235294117|
+------------------+



In [18]:
df.groupBy("median_income").avg("median_house_value") \
  .orderBy("avg(median_house_value)", ascending=False) \
  .show(10)

+-------------+-----------------------+
|median_income|avg(median_house_value)|
+-------------+-----------------------+
|      11.2866|               500001.0|
|      14.9009|               500001.0|
|       0.7025|               500001.0|
|       7.8647|               500001.0|
|      10.7582|               500001.0|
|       7.1669|               500001.0|
|       5.0222|               500001.0|
|      12.3804|               500001.0|
|       7.8521|               500001.0|
|       4.8482|               500001.0|
+-------------+-----------------------+
only showing top 10 rows

